In [33]:
from pynq import Overlay, MMIO
import pynq.lib as lib
import time
import numpy as np
from datetime import datetime
from pytz import timezone
overlay = Overlay("./kv_spi_6M_wrapper.bit")
AxiQspi = overlay.axi_quad_spi_0
overlay?

In [34]:
# Offset
# Interrupt Control Grouping
XSP_DGIER_OFFSET = 0x1C # Device global interrupt enable register
XSP_IISR_OFFSET = 0x20 # IP interrupt status register
XSP_IIER_OFFSET = 0x28 # IP interrupt enable register
# Core Grouping
XSP_SRR_OFFSET = 0x40 # Software reset register
XSP_CR_OFFSET = 0x60 # SPI control register
XSP_SR_OFFSET = 0x64 # SPI status register
XSP_DTR_OFFSET = 0x68 # SPI data transmit register.
XSP_DRR_OFFSET = 0x6C # SPI data receive register
XSP_SSR_OFFSET = 0x70 # SPI Slave select register
XSP_TFO_OFFSET = 0x74 # Transmit FIFO occupancy register
XSP_RFO_OFFSET = 0x78 # Receive FIFO occupancy register
XSP_REGISTERS = [0x40, 0x60, 0x64, 0x68, 0x6c, 0x70, 0x74, 0x78, 0x1c, 0x20, 0x28]

# Mask
XSP_SRR_RESET_MASK = 0x0A
XSP_SR_TX_EMPTY_MASK = 0x04
XSP_SR_TX_FULL_MASK = 0x08
## Crl mask
XSP_CR_TRANS_INHIBIT_MASK = 0x100 # Master Transaction Inhibit 0 = enable, 1 = disable
XSP_CR_LOOPBACK_MASK = 0x01 # Lookback mode: 0 = Normal operation, 1 = Loopback mode
XSP_CR_ENABLE_MASK = 0x02 # SPI system enable: 0 = SPI system disabled, 1 = SPI system enabled.
XSP_CR_MASTER_MODE_MASK = 0x04 # Set SPI mode: 0 = Slave configuration, 1 = Master configuration.
XSP_CR_CLK_POLARITY_MASK = 0x08 # Clock polarity
XSP_CR_CLK_PHASE_MASK = 0x10 # Clock phase
XSP_CR_TXFIFO_RESET_MASK = 0x20 # Transmit FIFO reset: 0 = Transmit FIFO normal operation, 1 = Reset transmit FIFO pointer.
XSP_CR_RXFIFO_RESET_MASK = 0x40 # Receive FIFO reset: 0 = Receive FIFO normal operation, 1 = Reset receive FIFO pointer.
XSP_CR_MANUAL_SS_MASK = 0x80 # Manual slave select assertion enable

SLAVE_NO_SELECTION = 0xFFFFFFFF

In [35]:
def cnfg(AxiQspi, clk_phase=0, clk_pol=0):
    print("Configure device")
    # Reset the SPI device
    AxiQspi.write(XSP_SRR_OFFSET, XSP_SRR_RESET_MASK)
    # Enable the transmit empty interrupt, which we use to determine progress on the transmission. 
    AxiQspi.write(XSP_IIER_OFFSET, XSP_SR_TX_EMPTY_MASK)
    # Disable the global IPIF interrupt
    AxiQspi.write(XSP_DGIER_OFFSET, 0)
    # Deselect the slave on the SPI bus
    AxiQspi.write(XSP_SSR_OFFSET, SLAVE_NO_SELECTION)
    # Disable the transmitter, enable Manual Slave Select Assertion, put SPI controller into master mode, and enable it
    ControlReg = AxiQspi.read(XSP_CR_OFFSET)
    ControlReg = ControlReg | XSP_CR_MASTER_MODE_MASK | XSP_CR_MANUAL_SS_MASK | XSP_CR_ENABLE_MASK | XSP_CR_TXFIFO_RESET_MASK | XSP_CR_RXFIFO_RESET_MASK
    AxiQspi.write(XSP_CR_OFFSET, ControlReg)
    
    ControlReg = AxiQspi.read(XSP_CR_OFFSET)
    ControlReg &= ~(XSP_CR_CLK_PHASE_MASK | XSP_CR_CLK_POLARITY_MASK)
    if clk_pol == 1:    # CPOL
        ControlReg |= XSP_CR_CLK_POLARITY_MASK
    if clk_phase == 1:  # CPHA
        ControlReg |= XSP_CR_CLK_PHASE_MASK
    AxiQspi.write(XSP_CR_OFFSET, ControlReg) #?
    CRRead = AxiQspi.read(XSP_CR_OFFSET)
    print(f"XSP_CR_OFFSET : {bin(CRRead)[2:].zfill(10)}")
    return 0
print("Done")

Done


In [36]:
cnfg(AxiQspi, clk_phase=0, clk_pol=0)

Configure device
XSP_CR_OFFSET : 0110000110


0

In [37]:
def xfer(B2Send, B2Recv, AxiQspi):
    """
    SPI Transfer function that sends and receives data in one 32-bit manner.
    Args:
        AxiQspi: SPI device object.
        B2Send : List of bytes to send .
        B2Recv : List to store received bytes .
    """

    # ----------------------
    # 1. 組合 4 Byte -> 32 bit，並一次寫入
    # ----------------------
    if len(B2Send) != 4 or B2Send.dtype != np.uint8:
        raise ValueError("B2Send must be a NumPy array of 4 uint8 bytes.")

    data32 = np.frombuffer(B2Send.tobytes(), dtype=np.uint32).byteswap()[0]

#     print(f"byte2Send: {B2Send} => 0x{data32:08X} (32-bit)")
    AxiQspi.write(XSP_DTR_OFFSET, int(data32))
#     StatusReg = AxiQspi.read(XSP_SR_OFFSET)
#     print(f"STAT_REG (二進制) : {bin(StatusReg)[2:].zfill(11)}")
    # 選擇 SPI Slave
    AxiQspi.write(XSP_SSR_OFFSET, 0xFFFFFFFE)

    # 清除 TRANS_INHIBIT_MASK
    ControlReg = AxiQspi.read(XSP_CR_OFFSET) & ~XSP_CR_TRANS_INHIBIT_MASK
    AxiQspi.write(XSP_CR_OFFSET, ControlReg)

    # 等待傳送完成 (TX FIFO empty)
    while (AxiQspi.read(XSP_SR_OFFSET) & XSP_SR_TX_EMPTY_MASK) == 0:
        pass

    # 讀取回一筆 (假設硬體會回傳 32 bit 到 RFO_OFFSET)
    temp_rfo = AxiQspi.read(XSP_RFO_OFFSET)
#     print(f"XSP_RFO_OFFSET: 0x{temp_rfo:08X}")

    # 傳送完後，重新設置 TRANS_INHIBIT_MASK
#     ControlReg = AxiQspi.read(XSP_CR_OFFSET)
    ControlReg |= XSP_CR_TRANS_INHIBIT_MASK
    AxiQspi.write(XSP_CR_OFFSET, ControlReg)

    # 取消所有 Slave
    AxiQspi.write(XSP_SSR_OFFSET, SLAVE_NO_SELECTION)
#     StatusReg = AxiQspi.read(XSP_SR_OFFSET)
#     print(f"STAT_REG (二進制) : {bin(StatusReg)[2:].zfill(11)}")
#     # ----------------------
#     # 2. 讀取 32 bit 並拆解成 4 個 8 bit
#     # ----------------------
#     print("ReadResponse")
#     RxFifoStatus = AxiQspi.read(XSP_SR_OFFSET) & 0x01
    
    # 清空 B2Recv (確保不會累加舊的資料)
    B2Recv = np.empty(0, dtype=np.uint8)
    while (AxiQspi.read(XSP_SR_OFFSET) & 0x01) == 0:
        temp32_drr = AxiQspi.read(XSP_DRR_OFFSET)
        # 使用 NumPy 快速轉換 32-bit 為 4-byte
        bytes_arr = np.frombuffer(temp32_drr.to_bytes(4, byteorder='big'), dtype=np.uint8)
        # 使用 np.concatenate 高效合併結果
        B2Recv = np.concatenate([B2Recv, bytes_arr])
        binary_representation = " ".join(f"{byte:08b}" for byte in B2Recv)
        print(f"B2Recv (Binary): {binary_representation}")

print("set xfer Done")

set xfer Done


In [83]:
iend = range(1, 9000)
for _ in iend:
    byte2Send = np.array([0xaa, 0xaa, 0xaa, 0xaa], dtype=np.uint8)
    byte2Recv = np.empty(4, dtype=np.uint8)
    xfer(byte2Send, byte2Recv, AxiQspi)

B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01100000 00111111 11111111 11111111
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01100000 00111111 11111111 11111111
B2Recv (Binary): 01000000 01100000 00111111 11111111
B2Recv (Binary): 01000000 01100000 00111111 11111111
B2Recv (Binary): 01100000 00111111 11111111 11111111
B2Recv (Binary): 01000000 01100000 00000000 00000000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01100000 00111111 11111111 11111111
B2Recv (Binary): 01000000 01100000 00111111 11111111
B2Recv (Binary): 01100000 00111111 11111111 11111111
B2Recv (Binary): 01100000 00111111 11111111 11

B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01100000 00111111 11111111 11111111
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01100000 00111111 11111111 11111111
B2Recv (Binary): 01000000 01100000 00000000 00000000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01100000 00111111 11111111
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01100000 00111111 11111111 11

B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11

B2Recv (Binary): 01100000 00111111 11111111 11111111
B2Recv (Binary): 01100000 00111111 11111111 11111111
B2Recv (Binary): 01100000 00111111 11111111 11111111
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01100000 00111111 11111111
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01100000 00111111 11111111 11111111
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01100000 00111111 11111111
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11110000
B2Recv (Binary): 01000000 01000011 00000010 11

In [76]:
byte2Send = np.array([0xaa, 0xaa, 0xaa, 0xaa], dtype=np.uint8)
byte2Recv = np.empty(4, dtype=np.uint8)
xfer(byte2Send, byte2Recv, AxiQspi)

B2Recv (Binary): 00000000 00000000 00000000 00000000
